In [2]:
import numpy as np
import pandas as pd
import requests
import re
from bs4 import BeautifulSoup
from urllib import response

#1. Collection des données

1.  Scrapping des infos sur la page d'accuiel et sur les rubriques informations sur le diabete

In [3]:
url = 'https://www.diabete.qc.ca/le-diabete-en-questions/'
response = requests.get(url)
if response.status_code == 200  :
    soup = BeautifulSoup(response.content, 'html.parser')
    my_dict = {}
    sections = soup.find_all('section', class_ = 'accordion')
    for section in sections :
        answers = []
        question = section.find('span', class_ = 'accordion__header__title').text
        results = section.find('div', class_ = 'accordion__sub-rows').find_all('p')
        for result in results :
            answers.append(result.text)
        my_dict[question] = ' '.join(answers)

df = pd.DataFrame(my_dict.items(), columns = ['questions', 'answers'])
df

,questions,answers
0,Qu'est-ce que le diabète ?,Le diabète est une maladie chronique qui ne se...
1,Quelles sont les complications du diabète ?,Les complications liées au diabète ont une ori...
2,Quels sont les symptômes ?,Les symptômes du diabète peuvent varier d’une ...
3,Quels sont les principaux types de diabète ?,Le diabète de type 1 Le diabète de type 1 se m...
4,Quels sont les facteurs de risques du diabète ...,Les causes du diabète de type 2 sont nombreuse...
5,Comment traite-t-on le diabète ?,Il existe plusieurs approches de traitement po...
6,Peut-on prévenir le diabète ?,On peut prévenir le diabète de type 2 ou en r...
7,Est-ce que le diabète se guérit ?,Le diabète est une maladie chronique incurable...


In [4]:
URL = 'https://www.diabete.qc.ca/le-diabete/informations-sur-le-diabete/'
URNs = [
    'quest-ce-que-le-diabete', 'facteurs-de-risque', 'symptomes',
    'depistage-et-diagnostic', 'diabete-de-type-1', 'diabete-de-type-2',
    'diabete-de-grossesse', 'prediabete', 'autres-types-de-diabete'
]
my_second_dict = {}
pattern = r'[\xa0\n]'

for URI in URNs :
    print(f'Getting resources for : {URI}...')
    try:
        response = requests.get(f'{URL}{URI}')
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        question = soup.find('h1')
        if question :
            question = question.text.strip()
        else :
            print(f"No <h1> tag found on page: {URI}")
            continue
        answer = soup.find('div', class_ = 'entry-content')
        if answer :
            answer = re.sub(pattern, ' ', answer.text.strip())
            my_second_dict[question] = answer
        else :
            print(f"No entry-content found on page : {URI}")
            continue
    except Exception as e :
        print(f"An error occurred with {URI} : {e}")

print(f"Total resources fetched : {len(my_second_dict)}")


Getting resources for : quest-ce-que-le-diabete...
Getting resources for : facteurs-de-risque...
Getting resources for : symptomes...
Getting resources for : depistage-et-diagnostic...
Getting resources for : diabete-de-type-1...
Getting resources for : diabete-de-type-2...
Getting resources for : diabete-de-grossesse...
Getting resources for : prediabete...
Getting resources for : autres-types-de-diabete...
Total resources fetched : 9


In [5]:
data = pd.concat([df, pd.DataFrame(my_second_dict.items(), columns = ['questions', 'answers'])], axis = 0)
data

,questions,answers
0,Qu'est-ce que le diabète ?,Le diabète est une maladie chronique qui ne se...
1,Quelles sont les complications du diabète ?,Les complications liées au diabète ont une ori...
2,Quels sont les symptômes ?,Les symptômes du diabète peuvent varier d’une ...
3,Quels sont les principaux types de diabète ?,Le diabète de type 1 Le diabète de type 1 se m...
4,Quels sont les facteurs de risques du diabète ...,Les causes du diabète de type 2 sont nombreuse...
5,Comment traite-t-on le diabète ?,Il existe plusieurs approches de traitement po...
6,Peut-on prévenir le diabète ?,On peut prévenir le diabète de type 2 ou en r...
7,Est-ce que le diabète se guérit ?,Le diabète est une maladie chronique incurable...
0,Qu’est-ce que le diabète ?,Le diabète est une maladie chronique qui ne se...
1,Facteurs de risque,Les causes du diabète de type 2 sont nombreuse...
